In [2]:
import kagglehub
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# 1. Download and load the new dataset
path = kagglehub.dataset_download("bhautikvekariya21/air-quality-dataset-indian-cities-2022-2025")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# 2. Map dataset columns
rename_map = {
    "city": "City",
    "datetime": "Date",
    "pm2_5_ugm3": "PM2.5",
    "pm10_ugm3": "PM10",
    "no2_ugm3": "NO2",
    "so2_ugm3": "SO2",
    "co_ugm3": "CO",
    "o3_ugm3": "O3",
    "humidity_percent": "Humidity",
    "wind_gusts_kmh": "Wind_Speed",
    "festival_period": "Festival",
    "crop_burning_season": "Crop_Burning",
    "aqi_category": "AQI_Bucket"
}
df = df.rename(columns=rename_map)

# Handle Temperature and numerical AQI targets
if "temp_2m" in df.columns: df = df.rename(columns={"temp_2m": "Temperature"})
elif "temperature" in df.columns: df = df.rename(columns={"temperature": "Temperature"})
else: df["Temperature"] = 25.0

aqi_num_cols = [c for c in df.columns if 'aqi' in c.lower() and c != 'AQI_Bucket']
if aqi_num_cols:
    df = df.rename(columns={aqi_num_cols[0]: "AQI"})
else:
    df["AQI"] = df["PM2.5"] * 1.5 

df = df.dropna(subset=["AQI", "AQI_Bucket"]).copy()

# 3. Extract Time Features
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year.fillna(2023).astype(int)
df['Month'] = df['Date'].dt.month.fillna(1).astype(int)
df['Day'] = df['Date'].dt.day.fillna(1).astype(int)

# 4. Encoders
city_encoder = LabelEncoder()
df['City_Encoded'] = city_encoder.fit_transform(df['City'].astype(str))

aqi_encoder = LabelEncoder()
df['Category_Encoded'] = aqi_encoder.fit_transform(df['AQI_Bucket'].astype(str))

# 5. Define Feature Groups
numerical_features = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'Temperature', 'Humidity', 'Wind_Speed']
context_features = ['Crop_Burning', 'Festival']
time_features = ['Year', 'Month', 'Day', 'City_Encoded']

for col in numerical_features:
    if col not in df.columns: df[col] = 0.0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median() if not df[col].dropna().empty else 0)

for col in context_features:
    if col not in df.columns: df[col] = 0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

X = df[numerical_features + context_features + time_features]
y_reg = pd.to_numeric(df['AQI'], errors='coerce')
y_clf = df['Category_Encoded']

valid_idx = X.dropna().index.intersection(y_reg.dropna().index)
X, y_reg, y_clf = X.loc[valid_idx], y_reg.loc[valid_idx], y_clf.loc[valid_idx]

# 6. Feature Scaling
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numerical_features] = scaler.fit_transform(X[numerical_features])

# 7. Train-Test Split
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_scaled, y_reg, y_clf, test_size=0.2, random_state=42
)

# 8. Train Models
print("Training Regressor...")
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train, y_reg_train)

print("Training Classifier...")
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_clf_train)

# 9. Save Artifacts
artifacts = {
    'regressor': rf_regressor,
    'classifier': rf_classifier,
    'scaler': scaler,
    'city_encoder': city_encoder,
    'aqi_encoder': aqi_encoder,
    'features': list(X.columns),
    'numerical_features': numerical_features
}

with open('aqi_models.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("Version 2.0 Artifacts saved successfully!")

Training Regressor...
Training Classifier...
Version 2.0 Artifacts saved successfully!


In [3]:
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score

# 1. Regressor Performance
reg_preds = rf_regressor.predict(X_test)
print(f"Mean Absolute Error (AQI points): {mean_absolute_error(y_reg_test, reg_preds):.2f}")
print(f"R-Squared Score: {r2_score(y_reg_test, reg_preds):.2f}")

# 2. Classifier Performance
clf_preds = rf_classifier.predict(X_test)
print(f"Category Accuracy: {accuracy_score(y_clf_test, clf_preds) * 100:.2f}%")

Mean Absolute Error (AQI points): 9.60
R-Squared Score: 0.91
Category Accuracy: 84.99%
